# 离线比较核函数

该 Notebook 使用模拟损失函数检验优化模块接口，不连接真机。比较真机算法时，使用相同的参数范围、初始观测和 shots 预算。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "execute":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in Platform_LQ or its execute directory")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from optimize.search import Search
rng = np.random.default_rng(42)
def true_loss(x):
    a, b = x
    return (a-.37)**2 + .4*(b-.68)**2 + .03*np.sin(18*a)*np.sin(10*b)
bounds = [[0, 1], [0, 1]]
initial = rng.uniform(0, 1, size=(6, 2))
noise_var = .005**2
traces = {}
for kernel in ("rbf", "matern32", "matern52"):
    measurement_rng = np.random.default_rng(123)
    search = Search(bounds, kernel=kernel, seed=42, initial_points=len(initial))
    for x in initial:
        search.tell(x, true_loss(x) + measurement_rng.normal(0, np.sqrt(noise_var)), noise_var)
    path = []
    for _ in range(18):
        x = search.ask()
        search.tell(x, true_loss(x) + measurement_rng.normal(0, np.sqrt(noise_var)), noise_var)
        path.append(min(true_loss(o.x) for o in search.observations))
    traces[kernel] = path
for name, values in traces.items():
    plt.plot(values, label=name)
plt.xlabel("Additional evaluations")
plt.ylabel("Best true loss (simulation only)")
plt.legend()
plt.grid(alpha=.3)
plt.show()
